# Advection-Diffusion: kappa/alpha/gamma Sweep

This notebook explores how the hyperparameters `kappa`, `alpha`, and `gamma` affect MESS estimation results. It sweeps over combinations of these values while keeping the observation indices fixed, and compares posterior samples for the skew-symmetric matrix $A$.

Key goals:
- Generate data for multiple (kappa, alpha, gamma) configurations.
- Run MESS with $M=5$ and uniform distance (no angular distance).
- Save chains to disk for reproducibility.
- Visualize posterior histograms for selected components of $A$ across configurations.

## Imports

In [ ]:
# Core imports and repo-local modules.
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from mess.data.advection_diffusion import generate_advection_diffusion_data
from mess.problems.advection_diffusion import AdvectionDiffusionToy, ij_to_k
from mess.algorithms.mess import mess_step

## Global Configuration
These settings control dimension, seeds, and default hyperparameters. Edit the sweep lists to explore additional combinations.

In [ ]:
# Seeds
seed_data = 0
seed_mcmc = 0

# Dimensions
dim = 4
num_params = dim * (dim - 1) // 2

# Hyperparameter sweep values
kappa_values = [0.02, 0.1]
alpha_values = [1, 2.0, 3.0]
gamma_values = [2, 3.0, 6.0]

# Fixed settings
sigma = 0.5  # Observation noise standard deviation
tau2 = 2.0  # Prior scale
a_mode = "nearest_neighbor"
use_prior_A = True  # If True, sample A from the prior

# Observation configuration (fixed for this sweep)
# all modes
obs_highest_freq = dim - 1
obs_bandwidth = dim
obs_config = "all_modes"
# central modes
obs_highest_freq = dim // 2
obs_bandwidth = dim // 2
obs_config = "central_modes"

# MCMC configuration
n_iters = 300000
burn_in = 2000
thin = 1
mess_M = 5

# Output locations
exp_dir = Path(repo_root) / "estimations" / "AD_toy_param_sweep" / f"dim{dim}_obs_{obs_config}_tau2{tau2}_sigma{sigma}_seed{seed_data}_Mmess{mess_M}_Niters{n_iters}"
exp_dir.mkdir(parents=True, exist_ok=True)

## Helper Functions
Utilities for building the problem, running MESS, saving results, and plotting histograms.

In [ ]:
def get_obs_indices(highest_freq, bandwidth, dim_value):
    bandwidth = max(1, min(bandwidth, dim_value))
    highest_freq = max(0, min(highest_freq, dim_value - 1))
    start = max(0, highest_freq - bandwidth + 1)
    return np.arange(start, highest_freq + 1, dtype=int)


def build_problem_for_params(kappa_value, alpha_value, gamma_value, use_prior_A=False):
    obs_indices = get_obs_indices(obs_highest_freq, obs_bandwidth, dim)
    a_mode_local = "prior" if use_prior_A else a_mode

    data = generate_advection_diffusion_data(
        dim=dim,
        kappa=kappa_value,
        sigma=sigma,
        obs_indices=obs_indices,
        alpha=alpha_value,
        gamma=gamma_value,
        tau2=tau2,
        a_mode=a_mode_local,
        seed=seed_data,
    )
    problem = AdvectionDiffusionToy(
        dim=dim,
        kappa=kappa_value,
        sigma=sigma,
        y=data['y'],
        obs_indices=obs_indices,
        g=data['g'],
        prior_diag=data['prior_diag'],
    )
    return data, problem, obs_indices


def run_mess_chain(problem, x0, n_steps, rng, M=10):
    chain = np.zeros((n_steps + 1, x0.shape[0]))
    chain[0] = x0.copy()
    x = x0.copy()
    for t in range(n_steps):
        x, _, _ = mess_step(x, problem, rng, M=M, use_lp=False)
        chain[t + 1] = x
    return chain


def save_chain(save_path, chain, metadata):
    np.savez_compressed(
        save_path,
        chain=chain,
        metadata=metadata,
    )


def get_component_labels(dim_value, comps):
    labels = []
    for k in comps:
        # Convert k back to (i, j) for readability.
        count = 0
        found = False
        for i in range(dim_value):
            for j in range(i + 1, dim_value):
                if count == k:
                    labels.append(f"a_{{{i}{j}}}")
                    found = True
                    break
                count += 1
            if found:
                break
        if not found:
            labels.append(f"param_{k}")
    return labels


def plot_posterior_histograms(chains_by_label, obs_indices_by_label, a_true, comps, bins=30):
    n_comp = len(comps)
    n_cols = 2
    n_rows = int(np.ceil(n_comp / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows), sharex=False)
    axes = np.array(axes).reshape(-1)
    labels = get_component_labels(dim, comps)
    for idx, comp in enumerate(comps):
        ax = axes[idx]
        for label, chain in chains_by_label.items():
            post = chain[burn_in::thin]
            ax.hist(post[:, comp], bins=bins, density=True, alpha=0.35, label=label)
            mean_val = float(np.mean(post[:, comp]))
            ax.axvline(mean_val, color='black', linestyle=':', linewidth=1.0)
        ax.axvline(a_true[comp], color='red', linestyle='--', linewidth=1.3)
        ax.set_title(labels[idx])
        ax.grid(alpha=0.3)
    for ax in axes[n_comp:]:
        ax.axis('off')
    # obs_summary = ' | '.join([f"{label}: {obs.tolist()}" for label, obs in obs_indices_by_label.items()])
    # fig.suptitle(f"Observed indices: {obs_summary}", fontsize=10)
    axes[0].legend(loc='upper right')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    return fig


def plot_hist_by_config(chains_by_label, obs_indices_by_label, datasets, comps, bins=30):
    labels = get_component_labels(dim, comps)
    figs = []
    for label, chain in chains_by_label.items():
        data = datasets[label]
        a_true = data['a_true']
        n_comp = len(comps)
        n_cols = 2
        n_rows = int(np.ceil(n_comp / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows), sharex=False)
        axes = np.array(axes).reshape(-1)
        post = chain[burn_in::thin]
        for idx, comp in enumerate(comps):
            ax = axes[idx]
            ax.hist(post[:, comp], bins=bins, density=True, alpha=0.5, color='steelblue')
            mean_val = float(np.mean(post[:, comp]))
            ax.axvline(mean_val, color='black', linestyle=':', linewidth=1.0)
            ax.axvline(a_true[comp], color='red', linestyle='--', linewidth=1.3)
            ax.set_title(labels[idx])
            ax.grid(alpha=0.3)
        for ax in axes[n_comp:]:
            ax.axis('off')
        obs_indices = obs_indices_by_label.get(label)
        if obs_indices is not None:
            title = f"{label} | obs_indices={obs_indices.tolist()}"
        else:
            title = label
        fig.suptitle(title, fontsize=10)
        plt.tight_layout(rect=[0, 0, 1, 0.93])
        figs.append(fig)
    return figs


def plot_bias_image_by_config(chains_by_label, obs_indices_by_label, datasets, comps):
    fig, ax = plt.subplots(figsize=(10, 4))
    for label, chain in chains_by_label.items():
        data = datasets[label]
        a_true = data['a_true']
        post = chain[burn_in::thin]
        bias_config = np.mean(post[:, comps], axis=0) - a_true[comps]
        ax.plot(bias_config, label=label)
    ax.axhline(0.0, color='black', linewidth=1.0, linestyle='--')
    ax.set_title('Posterior mean bias by config')
    ax.set_xlabel('Parameter index')
    ax.set_ylabel('Posterior mean - true')
    ax.grid(alpha=0.3)
    ax.legend(loc='best')
    plt.tight_layout()
    return fig


def format_value(value):
    return str(value).replace('.', 'p')

## Run Hyperparameter Sweep
This runs MESS for each (kappa, alpha, gamma) combination, saves the chain, and stores results in memory for plotting.

In [ ]:
param_configs = [
    {'kappa': kappa, 'alpha': alpha, 'gamma': gamma}
    for kappa in kappa_values
    for alpha in alpha_values
    for gamma in gamma_values
]
param_configs

In [ ]:
chains_by_label = {}
true_by_label = {}
obs_indices_by_label = {}
datasets_by_label = {}

for config in param_configs:
    kappa_value = config['kappa']
    alpha_value = config['alpha']
    gamma_value = config['gamma']

    data, problem, obs_indices = build_problem_for_params(
        kappa_value, alpha_value, gamma_value, use_prior_A=use_prior_A
    )
    rng = np.random.default_rng(seed_mcmc)

    label = f"kappa={kappa_value}, alpha={alpha_value}, gamma={gamma_value}"
    save_path = exp_dir / (
        f"mess_{mess_M}_N{n_iters}_k{format_value(kappa_value)}"
        f"_a{format_value(alpha_value)}_g{format_value(gamma_value)}.npz"
    )

    metadata = {
        'dim': dim,
        'obs_highest_freq': obs_highest_freq,
        'obs_bandwidth': obs_bandwidth,
        'obs_indices': obs_indices.tolist(),
        'seed_data': seed_data,
        'seed_mcmc': seed_mcmc,
        'n_iters': n_iters,
        'burn_in': burn_in,
        'thin': thin,
        'mess_M': mess_M,
        'hyperparameters': {
            'kappa': kappa_value,
            'sigma': sigma,
            'alpha': alpha_value,
            'gamma': gamma_value,
            'tau2': tau2,
            'a_mode': 'prior' if use_prior_A else a_mode,
            'use_prior_A': use_prior_A,
        },
    }

    if save_path.exists():
        print(f"Chain already exists at {save_path}, skipping MCMC run.")
        chain = np.load(save_path)['chain']
    else:
        print(
            "Running MESS chain for "
            f"kappa={kappa_value}, alpha={alpha_value}, gamma={gamma_value} "
            f"with obs_indices={obs_indices}..."
        )
        chain = run_mess_chain(problem, data['a_init'], n_iters, rng, M=mess_M)
        save_chain(save_path, chain, metadata)
        print(f"Saved chain to {save_path}")

    chains_by_label[label] = chain
    true_by_label[label] = data['a_true']
    obs_indices_by_label[label] = obs_indices
    datasets_by_label[label] = data

print("Finished sweep.")

## Posterior Histograms for Selected Components
Compare posterior samples across hyperparameter configurations for chosen $A$ components.

In [ ]:
# Select components to visualize (indices in the vectorized upper triangle).
comps = np.arange(num_params)

# Use the first configuration's truth as reference in plots.
first_label = list(true_by_label.keys())[0]
a_true = true_by_label[first_label]

fig = plot_posterior_histograms(chains_by_label, obs_indices_by_label, a_true, comps, bins=30)
fig.savefig(exp_dir / "posterior_histograms_param_sweep.png", dpi=300)
plt.show()

## Optional: Inspect Observed Indices
This cell prints the observation indices for each configuration so you can confirm the slice definitions.

In [ ]:
for label, obs_indices in obs_indices_by_label.items():
    print(label, "obs_indices=", obs_indices.tolist())

## Posterior Histograms by Config
Plot histograms separately for each (kappa, alpha, gamma) configuration.

In [ ]:
# Separate histograms per parameter config (no overlays).
comps = np.arange(num_params)
figs_by_config = plot_hist_by_config(
    chains_by_label, obs_indices_by_label, datasets_by_label, comps
 )
plt.show()

## Posterior Marginals and Pairwise Densities
Generate pairwise density grids for each parameter configuration.

In [ ]:
# extract key, val pairs 
for key in datasets_by_label.keys():
    dataset = datasets_by_label[key]
    a_tr = dataset['a_true']
    prior_di = dataset['prior_diag']
    print(a_tr)
    print(prior_di)
    print()

In [ ]:
from mess.plotting.diagnostics import write_csv, plot_timeseries, make_hist_grid_comps

panel_comps = np.arange(num_params)
labels_chains = [
    f"kappa={cfg['kappa']}, alpha={cfg['alpha']}, gamma={cfg['gamma']}"
    for cfg in param_configs
]
chains_plot = {label: chains_by_label[label] for label in labels_chains}
label_map = dict(zip(panel_comps, get_component_labels(dim, panel_comps)))
dataset
covariances = {}
kappas = {}
alphas = {}
gammas = {}
tau2s = {}
sigmas = {}
true_vals = {}

for cfg in param_configs:
    kappa_value = cfg['kappa']
    alpha_value = cfg['alpha']
    gamma_value = cfg['gamma']
    lab = f"kappa={kappa_value}, alpha={alpha_value}, gamma={gamma_value}"
    data = datasets_by_label[lab]
    covariances[lab] = data['prior_diag']
    true_vals[lab] = data['a_true']
    kappas[lab] = data['kappa']
    alphas[lab] = data['alpha']
    gammas[lab] = data['gamma']
    tau2s[lab] = data['tau2']
    sigmas[lab] = data['sigma']
    print(f"\nconfig: {lab}, \nprior cov: {covariances[lab]}, \ntrue vals: {true_vals[lab]}\n")

plot_title = "Posterior Marginals and Pairwise Densities"
font_size = 12

for label, est_chain in chains_plot.items():
    R = sigmas[label] / 4
    dr = R / 30
    fig = make_hist_grid_comps(
        R,
        dr,
        est_chain,
        panel_comps,
        None,
        C=np.diag(covariances[label]),
        beta=0.95,
        hide_plot=False,
        label_map=label_map,
        font_size=font_size,
        title=(
            f"{label}; kappa={kappas[label]}, alpha={alphas[label]}, "
            f"gamma={gammas[label]}, tau2={tau2s[label]}, sigma={sigmas[label]}"
        ),
        figsize=(15, 15),
        true_values=true_vals[label],
    )

In [ ]:
# the z sample (driven by the seed) makes a big difference in the posterior (since it determines the true A and thus the data)
# the seed 0 gives more interesting posteriors


## Targeted Runs (Seed 0)
Run two specific (alpha, gamma) configurations with kappa=0.05, sigma=0.5, tau2=2.0.

In [ ]:
# Seeds
seed_data = 0
seed_mcmc = 0

# Dimensions
dim = 10
num_params = dim * (dim - 1) // 2

# Hyperparameter sweep values
kappa = 0.02

# Fixed settings
sigma = 0.5  # Observation noise standard deviation
tau2 = 2.0  # Prior scale
a_mode = "nearest_neighbor"
use_prior_A = True  # If True, sample A from the prior

# Observation configuration (fixed for this sweep)
# all modes
# obs_highest_freq = dim - 1
# obs_bandwidth = dim
# obs_config = "all_modes"
# central modes
obs_highest_freq = 6
obs_bandwidth = 3
obs_config = "central_modes"

# MCMC configuration
n_iters = 150000
burn_in = 1000
thin = 1
mess_M = 20

# Output locations
exp_dir = Path(repo_root) / "estimations" / "AD_toy_param_sweep" / f"dim{dim}_inversecrime{use_prior_A}_obs_{obs_config}_tau2{tau2}_sigma{sigma}_seed{seed_data}_Mmess{mess_M}_Niters{n_iters}"
exp_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
param_configs = [
    # {'kappa': 0.01, 'alpha': 2, 'gamma': 3},
    # {'kappa': 0.01, 'alpha': 3, 'gamma': 2},
    # {'kappa': 0.01, 'alpha': 2, 'gamma': 1},
    # {'kappa': 0.02, 'alpha': 3, 'gamma': 1},
    {'kappa': 0.02, 'alpha': 3, 'gamma': 2},
    # {'kappa': 0.02, 'alpha': 2, 'gamma': 1},
]
param_configs

In [ ]:
chains_by_label = {}
true_by_label = {}
obs_indices_by_label = {}
datasets_by_label = {}

for config in param_configs:
    kappa_value = config['kappa']
    alpha_value = config['alpha']
    gamma_value = config['gamma']

    data, problem, obs_indices = build_problem_for_params(
        kappa_value, alpha_value, gamma_value, use_prior_A=use_prior_A
    )
    rng = np.random.default_rng(seed_mcmc)

    label = f"kappa={kappa_value}, alpha={alpha_value}, gamma={gamma_value}"
    save_path = exp_dir / (
        f"mess_{mess_M}_N{n_iters}_k{format_value(kappa_value)}"
        f"_a{format_value(alpha_value)}_g{format_value(gamma_value)}.npz"
    )

    metadata = {
        'dim': dim,
        'obs_highest_freq': obs_highest_freq,
        'obs_bandwidth': obs_bandwidth,
        'obs_indices': obs_indices.tolist(),
        'seed_data': seed_data,
        'seed_mcmc': seed_mcmc,
        'n_iters': n_iters,
        'burn_in': burn_in,
        'thin': thin,
        'mess_M': mess_M,
        'hyperparameters': {
            'kappa': kappa_value,
            'sigma': sigma,
            'alpha': alpha_value,
            'gamma': gamma_value,
            'tau2': tau2,
            'a_mode': 'prior' if use_prior_A else a_mode,
            'use_prior_A': use_prior_A,
        },
    }

    if save_path.exists():
        print(f"Chain already exists at {save_path}, skipping MCMC run.")
        chain = np.load(save_path)['chain']
    else:
        print(
            "Running MESS chain for "
            f"kappa={kappa_value}, alpha={alpha_value}, gamma={gamma_value} "
            f"with obs_indices={obs_indices}..."
        )
        chain = run_mess_chain(problem, data['a_init'], n_iters, rng, M=mess_M)
        save_chain(save_path, chain, metadata)
        print(f"Saved chain to {save_path}")

    chains_by_label[label] = chain
    true_by_label[label] = data['a_true']
    obs_indices_by_label[label] = obs_indices
    datasets_by_label[label] = data

print("Finished sweep.")

In [ ]:
from mess.plotting.diagnostics import write_csv, plot_timeseries, make_hist_grid_comps

panel_comps = [0, 1, 2, 3, 10, 15, 25, 30, ij_to_k(0, dim - 1, dim), num_params - 2, num_params - 1]
panel_comps = [0, 1, 2, 3, 9, 15, 25, 30, num_params - 2, num_params - 1]
panel_comps = [0, 1, 2, 3, 9, 10, 17]
# ind_plot = 4
# panel_comps = np.arange(10*ind_plot, 10*(ind_plot+1))
# panel_comps = np.arange(40, 45)
labels_chains = [
    f"kappa={cfg['kappa']}, alpha={cfg['alpha']}, gamma={cfg['gamma']}"
    for cfg in param_configs
]
chains_plot = {label: chains_by_label[label] for label in labels_chains}
label_map = {
    comp: rf"${label}$"
    for comp, label in zip(panel_comps, get_component_labels(dim, panel_comps))
}
dataset
covariances = {}
kappas = {}
alphas = {}
gammas = {}
tau2s = {}
sigmas = {}
true_vals = {}

for cfg in param_configs:
    kappa_value = cfg['kappa']
    alpha_value = cfg['alpha']
    gamma_value = cfg['gamma']
    lab = f"kappa={kappa_value}, alpha={alpha_value}, gamma={gamma_value}"
    data = datasets_by_label[lab]
    covariances[lab] = data['prior_diag']
    true_vals[lab] = data['a_true']
    kappas[lab] = data['kappa']
    alphas[lab] = data['alpha']
    gammas[lab] = data['gamma']
    tau2s[lab] = data['tau2']
    sigmas[lab] = data['sigma']
    print(f"\nconfig: {lab}, \nprior cov: {covariances[lab]}, \ntrue vals: {true_vals[lab]}\n")

plot_title = "Posterior Marginals and Pairwise Densities"
label_font_size = 18
tick_font_size = 14
title_font_size = 16

for label, est_chain in chains_plot.items():
    R = sigmas[label]/2
    dr = R / 80
    title = (
        rf"$\kappa={kappas[label]}, \alpha={alphas[label]}, \gamma={gammas[label]}, "
        rf"\tau^2={tau2s[label]}, \sigma={sigmas[label]}$"
    )
    fig = make_hist_grid_comps(
        R,
        dr,
        est_chain,
        panel_comps,
        None,
        C=np.diag(covariances[label]),
        beta=0.95,
        hide_plot=False,
        label_map=label_map,
        font_size=label_font_size,
        title=title,
        figsize=(15, 15),
        true_values=true_vals[label],
    )
    fig.suptitle(title, fontsize=title_font_size)
    for ax in fig.get_axes():
        ax.tick_params(axis="x", labelsize=tick_font_size)
        ax.set_xlabel(ax.get_xlabel(), fontsize=label_font_size)
        ax.tick_params(axis="y", labelsize=tick_font_size)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    # fig.savefig(exp_dir / f"posterior_grid_comps_{label.replace(', ', '_').replace('=', '')}.jpg", dpi=300)
